In [1]:
import psycopg2
import pandas as pd
from psycopg2.extras import execute_values
import os
from dotenv import load_dotenv


In [62]:
load_dotenv(dotenv_path=r"C:\Users\user\OneDrive\Desktop\HEALTH\.env")

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    sslmode="require"  
)

cur = conn.cursor()



In [63]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS patients (
        id                 SERIAL PRIMARY KEY,
        age                INTEGER CHECK (age BETWEEN 0 AND 120),
        gender             VARCHAR(10) NOT NULL,
        blood_type         VARCHAR(5),
        medical_condition  VARCHAR(100),
        admission_type     VARCHAR(50),
        medication         VARCHAR(100),
        billing_amount     NUMERIC(12,2) CHECK (billing_amount >= 0),
        test_results       VARCHAR(20) CHECK (test_results IN ('Normal','Abnormal','Inconclusive')),
        created_at         TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        UNIQUE (age, gender, blood_type, medical_condition, billing_amount)
    )
""")
conn.commit()
print("Table created ✓")

Table created ✓


In [64]:
conn.rollback()
print("Rollback done ✓")

Rollback done ✓


In [65]:

query = "SELECT * FROM patients;"
df = pd.read_sql(query, conn)

C:\Users\user\AppData\Local\Temp\ipykernel_15552\1921961917.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [66]:
print(df.columns)
print(df.head())

Index(['id', 'age', 'gender', 'blood_type', 'medical_condition',
       'admission_type', 'medication', 'billing_amount', 'test_results',
       'created_at'],
      dtype='str')
    id  age  gender blood_type medical_condition admission_type   medication  \
0  134   30    Male         B-            Cancer         Urgent  Paracetamol   
1  135   62    Male         A+           Obesity      Emergency    Ibuprofen   
2  136   76  Female         A-           Obesity      Emergency      Aspirin   
3  137   28  Female         O+          Diabetes       Elective    Ibuprofen   
4  138   43  Female        AB+            Cancer         Urgent   Penicillin   

   billing_amount  test_results                 created_at  
0        18856.28        Normal 2026-04-20 07:11:57.198467  
1        33643.33  Inconclusive 2026-04-20 07:11:57.198467  
2        27955.10        Normal 2026-04-20 07:11:57.198467  
3        37909.78      Abnormal 2026-04-20 07:11:57.198467  
4        14238.32      Abnormal 202

In [67]:
# df = pd.read_csv("data\cleaned_data.csv")
df["billing_amount"] = df["billing_amount"].abs()

In [68]:
# list to tuples for batch insertion

from psycopg2.extras import execute_values

records = [
    (
        row["age"],
        row["gender"],
        row["blood_type"],
        row["medical_condition"],
        row["admission_type"],
        row["medication"],
        round(row["billing_amount"], 2),
        row["test_results"],
        row["created_at"]
    )
    for _, row in df.iterrows()
]

execute_values(cur, """
    INSERT INTO patients (
        age, gender, blood_type, medical_condition,
        admission_type, medication, billing_amount, test_results, created_at
    )
    VALUES %s
    ON CONFLICT (age, gender, blood_type, medical_condition, billing_amount)
    DO NOTHING
""", records, page_size=500)

conn.commit()
print(f"Inserted {cur.rowcount} rows ✓")

Inserted 0 rows ✓


In [69]:
# verify insertion
cur.execute("SELECT test_results, COUNT(*) FROM patients GROUP BY test_results")
for row in cur.fetchall():
    print(row)

('Abnormal', 18437)
('Inconclusive', 18198)
('Normal', 18331)


In [70]:
cur.close()
conn.close()
print("Connection closed ✓")

Connection closed ✓
